The ABS-EE autoloan panel contains every loan-level filing submitted to the SEC under
Regulation AB II. In April 2026, that's **5.4 million active loans** across 171 trusts,
originated by everyone from GM Financial and Nissan to Carvana and Exeter.

This note looks at the credit quality side of that dataset: who's borrowing,
how creditworthy they are, and how that varies across sponsors and vehicle types.

Three metrics:
- **FICO** — obligor credit score at origination
- **LTV** — original loan amount ÷ vehicle value at origination (lower = better collateral coverage)
- **Credit tier** — FICO buckets that roughly map to prime, near-prime, subprime, and deep sub

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BLUE  = "#4A6FA5"
AMBER = "#C8964A"
RED   = "#C86A50"
TEAL  = "#5A8A8A"
GREEN = "#7EA870"
GREY  = "#D4CFC4"
BG    = "#FAF9F5"
TEXT  = "#1C1C1E"
MUTED = "#8A8A8A"

TIER_COLORS = {
    "750+  Prime+":      BLUE,
    "700-749  Prime":    TEAL,
    "660-699  Near-Prime": GREEN,
    "620-659  Subprime": AMBER,
    "<620  Deep Sub":    RED,
}

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   BG,
    "axes.edgecolor":   MUTED,
    "axes.labelcolor":  TEXT,
    "xtick.color":      TEXT,
    "ytick.color":      TEXT,
    "text.color":       TEXT,
    "font.family":      "sans-serif",
    "font.size":        11,
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

PERIOD = "2026-04"
OUTPUTS = Path("../../analysis/outputs")

by_deal = pd.read_parquet(OUTPUTS / f"fico_by_deal_{PERIOD}.parquet")
by_make = pd.read_parquet(OUTPUTS / f"fico_by_make_{PERIOD}.parquet")
by_tier = pd.read_parquet(OUTPUTS / f"fico_by_tier_{PERIOD}.parquet")

## Credit tier mix

The ABS-EE universe is not all subprime. The largest share of loans in April 2026 are
prime-quality borrowers — but the deep subprime shelf is substantial and worth watching.

In [ ]:
tier_totals = (
    by_tier.groupby("tier")[["loan_count", "orig_bal_mm"]]
    .sum()
    .reset_index()
    .sort_values("tier")
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), facecolor=BG)
fig.suptitle(f"ABS-EE Autoloan Credit Tier Mix — {PERIOD}", fontsize=13, color=TEXT, y=1.01)

colors = [TIER_COLORS.get(t, GREY) for t in tier_totals["tier"]]

axes[0].barh(tier_totals["tier"], tier_totals["loan_count"] / 1e6, color=colors)
axes[0].set_xlabel("Loans (millions)")
axes[0].set_title("By loan count")

axes[1].barh(tier_totals["tier"], tier_totals["orig_bal_mm"] / 1e3, color=colors)
axes[1].set_xlabel("Original balance ($B)")
axes[1].set_title("By original balance")

plt.tight_layout()
plt.show()

## LTV by credit tier

Prime borrowers tend to borrow less relative to vehicle value. Deep subprime borrowers
often have LTVs above 1.0 — the loan exceeds the vehicle's value at origination.
That's a meaningful signal: when a subprime borrower defaults and the vehicle is
repossessed, the collateral may not fully cover the outstanding balance.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), facecolor=BG)

tier_ltv = (
    by_tier.groupby("tier")["avg_ltv"]
    .mean()
    .reset_index()
    .sort_values("tier")
)

colors = [TIER_COLORS.get(t, GREY) for t in tier_ltv["tier"]]
bars = ax.barh(tier_ltv["tier"], tier_ltv["avg_ltv"], color=colors)
ax.axvline(1.0, color=MUTED, linestyle="--", linewidth=1, label="LTV = 1.0")
ax.set_xlabel("Average LTV at origination")
ax.set_title(f"LTV by credit tier — {PERIOD}")
ax.legend()

for bar, val in zip(bars, tier_ltv["avg_ltv"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=9, color=TEXT)

plt.tight_layout()
plt.show()

## Which sponsors run the primest shelves?

Captive finance arms (GM Financial, Nissan, BMW) consistently originate
higher-FICO, lower-LTV pools. Specialty lenders like Exeter, Bridgecrest, and
Carvana run deeper into the subprime spectrum — that's the business model.

In [ ]:
top_deals = by_deal.nlargest(20, "loan_count").sort_values("median_fico")

fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG)

scatter = ax.scatter(
    top_deals["median_fico"],
    top_deals["avg_ltv"],
    s=top_deals["loan_count"] / 800,
    c=top_deals["median_fico"],
    cmap="RdYlGn",
    alpha=0.85,
    edgecolors=MUTED,
    linewidths=0.5,
)

for _, row in top_deals.iterrows():
    label = row["sponsor"][:22] if isinstance(row["sponsor"], str) else ""
    ax.annotate(label, (row["median_fico"], row["avg_ltv"]),
                fontsize=7.5, color=TEXT,
                xytext=(4, 2), textcoords="offset points")

ax.axhline(1.0, color=MUTED, linestyle="--", linewidth=1, alpha=0.6)
ax.set_xlabel("Median FICO")
ax.set_ylabel("Average LTV")
ax.set_title(f"Top 20 deals by loan count — FICO vs LTV — {PERIOD}\n(bubble size = loan count)")
plt.colorbar(scatter, ax=ax, label="Median FICO")
plt.tight_layout()
plt.show()

## FICO by vehicle make

Luxury brands (BMW, Audi, Mercedes) unsurprisingly attract higher-FICO borrowers.
Volume brands vary more — the same Honda Civic gets financed by a 780-FICO GM Financial
customer and a 580-FICO Exeter customer.

In [ ]:
top_makes = (
    by_make.groupby("make")
    .agg(loan_count=("loan_count", "sum"), median_fico=("median_fico", "mean"), avg_ltv=("avg_ltv", "mean"))
    .reset_index()
    .nlargest(20, "loan_count")
    .sort_values("median_fico", ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 5.5), facecolor=BG)

norm = plt.Normalize(top_makes["median_fico"].min(), top_makes["median_fico"].max())
colors = plt.cm.RdYlGn(norm(top_makes["median_fico"]))

ax.barh(top_makes["make"], top_makes["median_fico"], color=colors)
ax.set_xlabel("Median FICO at origination")
ax.set_title(f"Top 20 vehicle makes by loan count — median FICO — {PERIOD}")
ax.set_xlim(550, 820)

for _, row in top_makes.iterrows():
    ax.text(row["median_fico"] + 1, row["make"],
            f"{row['median_fico']:.0f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

---

## Notes on the data

- **Coverage**: FICO is not reported by all trusts. ~3.5% of loans in this period have no
  credit score on file. Those loans are excluded from the charts above.
- **Schema variation**: Different securitization sponsors use slightly different field names
  in their SEC filings. The `union_by_name` merge handles this but means some
  trust-specific fields may be missing for non-reporting trusts.
- **FICO timing**: The score is recorded at origination, not at the reporting period.
  Older loans may have materially different current credit profiles.
- **Source**: SEC EDGAR Form ABS-EE, Schedule AL — autoloan asset class.
  Reporting period ending 2026-03-31 (filed April 2026).